## Stewart platform

## Inverse Kinematics

In [1]:
import sympy as sp
import numpy as np

r_m, r_f = sp.symbols('r_m r_f', positive=True, real=True)
t_m, t_f = sp.symbols('theta_m theta_f', real=True)
X, Y, Z, alpha, beta, gamma = sp.symbols('P_x P_y P_z alpha beta gamma', real=True)

# Rotation matrices
def rotX(theta):
    return sp.Matrix([
        [1, 0, 0],
        [0, sp.cos(theta), -sp.sin(theta)],
        [0, sp.sin(theta),  sp.cos(theta)]
    ])
def rotY(theta):
    return sp.Matrix([
        [sp.cos(theta), 0, sp.sin(theta)],
        [0, 1, 0],
        [-sp.sin(theta), 0, sp.cos(theta)]
    ])
def rotZ(theta):
    return sp.Matrix([
        [sp.cos(theta), -sp.sin(theta), 0],
        [sp.sin(theta), sp.cos(theta), 0],
        [0, 0, 1]
    ])
def clean(expr):
    # expr = sp.expand_trig(expr)
    # expr = sp.together(expr)
    expr = sp.cancel(expr)
    expr = sp.trigsimp(expr)
    expr = sp.simplify(expr)
    return expr
pi = sp.pi

In [2]:

# Home positions

# fixed base (O_r_Fi)
F_1 = sp.Matrix([   r_f * sp.cos(pi/3 - t_f/2),      r_f * sp.sin(pi/3 - t_f/2),   0])
F_2 = sp.Matrix([   r_f * sp.cos(pi/3 + t_f/2),      r_f * sp.sin(pi/3 + t_f/2),   0])
F_3 = sp.Matrix([ r_f * sp.cos(3*pi/3 - t_f/2),    r_f * sp.sin(3*pi/3 - t_f/2),   0])
F_4 = sp.Matrix([ r_f * sp.cos(3*pi/3 + t_f/2),    r_f * sp.sin(3*pi/3 + t_f/2),   0])
F_5 = sp.Matrix([ r_f * sp.cos(5*pi/3 - t_f/2),    r_f * sp.sin(5*pi/3 - t_f/2),   0])
F_6 = sp.Matrix([ r_f * sp.cos(5*pi/3 + t_f/2),    r_f * sp.sin(5*pi/3 + t_f/2),   0])

F = [F_1, F_2, F_3, F_4, F_5, F_6]

# moving platform (O1_r_Mi)
M_1 = sp.Matrix([   r_m * sp.cos(pi/3 - t_m/2),      r_m * sp.sin(pi/3 - t_m/2),   0])
M_2 = sp.Matrix([   r_m * sp.cos(pi/3 + t_m/2),      r_m * sp.sin(pi/3 + t_m/2),   0])
M_3 = sp.Matrix([ r_m * sp.cos(3*pi/3 - t_m/2),    r_m * sp.sin(3*pi/3 - t_m/2),   0])
M_4 = sp.Matrix([ r_m * sp.cos(3*pi/3 + t_m/2),    r_m * sp.sin(3*pi/3 + t_m/2),   0])
M_5 = sp.Matrix([ r_m * sp.cos(5*pi/3 - t_m/2),    r_m * sp.sin(5*pi/3 - t_m/2),   0])
M_6 = sp.Matrix([ r_m * sp.cos(5*pi/3 + t_m/2),    r_m * sp.sin(5*pi/3 + t_m/2),   0])

M = [M_1, M_2, M_3, M_4, M_5, M_6]

# Position of moving platform w.r.t fixed base
P = sp.Matrix([X, Y, Z])

In [3]:
np.size(P)

3

### Rotation of platform w.r.t base

In [4]:

base_R_platform = rotZ(gamma) * rotY(beta) * rotX(alpha)
base_R_platform

Matrix([
[cos(beta)*cos(gamma), sin(alpha)*sin(beta)*cos(gamma) - sin(gamma)*cos(alpha),  sin(alpha)*sin(gamma) + sin(beta)*cos(alpha)*cos(gamma)],
[sin(gamma)*cos(beta), sin(alpha)*sin(beta)*sin(gamma) + cos(alpha)*cos(gamma), -sin(alpha)*cos(gamma) + sin(beta)*sin(gamma)*cos(alpha)],
[          -sin(beta),                                    sin(alpha)*cos(beta),                                     cos(alpha)*cos(beta)]])

### Link vectors

In [5]:
# L = sp.Matrix([
#     base_R_platform @ M[i] + P - F[i]
#     for i in range(6)
# ])

# L = L.reshape(6, 3)
# L = clean(L)
# L

L1 = base_R_platform * M_1 + P - F_1
L2 = base_R_platform * M_2 + P - F_2
L3 = base_R_platform * M_3 + P - F_3
L4 = base_R_platform * M_4 + P - F_4
L5 = base_R_platform * M_5 + P - F_5
L6 = base_R_platform * M_6 + P - F_6

L = sp.Matrix([L1, L2, L3, L4, L5, L6])
L.reshape(6,3)

Matrix([
[P_x - r_f*sin(theta_f/2 + pi/6) + r_m*(sin(alpha)*sin(beta)*cos(gamma) - sin(gamma)*cos(alpha))*cos(theta_m/2 + pi/6) + r_m*sin(theta_m/2 + pi/6)*cos(beta)*cos(gamma), P_y - r_f*cos(theta_f/2 + pi/6) + r_m*(sin(alpha)*sin(beta)*sin(gamma) + cos(alpha)*cos(gamma))*cos(theta_m/2 + pi/6) + r_m*sin(gamma)*sin(theta_m/2 + pi/6)*cos(beta), P_z + r_m*sin(alpha)*cos(beta)*cos(theta_m/2 + pi/6) - r_m*sin(beta)*sin(theta_m/2 + pi/6)],
[P_x - r_f*cos(theta_f/2 + pi/3) + r_m*(sin(alpha)*sin(beta)*cos(gamma) - sin(gamma)*cos(alpha))*sin(theta_m/2 + pi/3) + r_m*cos(beta)*cos(gamma)*cos(theta_m/2 + pi/3), P_y - r_f*sin(theta_f/2 + pi/3) + r_m*(sin(alpha)*sin(beta)*sin(gamma) + cos(alpha)*cos(gamma))*sin(theta_m/2 + pi/3) + r_m*sin(gamma)*cos(beta)*cos(theta_m/2 + pi/3), P_z + r_m*sin(alpha)*sin(theta_m/2 + pi/3)*cos(beta) - r_m*sin(beta)*cos(theta_m/2 + pi/3)],
[                     P_x + r_f*cos(theta_f/2) + r_m*(sin(alpha)*sin(beta)*cos(gamma) - sin(gamma)*cos(alpha))*sin(theta_m/2) - r_m

### Loop closure equations

In [6]:
# l = sp.Matrix([
#     sp.sqrt(L.row(i).dot(L.row(i)))
#     for i in range(6)
# ])

# l = clean(l)

l_1 = sp.sqrt(L1.dot(L1))
l_2 = sp.sqrt(L2.dot(L2))
l_3 = sp.sqrt(L3.dot(L3))
l_4 = sp.sqrt(L4.dot(L4))
l_5 = sp.sqrt(L5.dot(L5))
l_6 = sp.sqrt(L6.dot(L6))

l = sp.Matrix([l_1, l_2, l_3, l_4, l_5, l_6])
l


Matrix([
[sqrt((P_z + r_m*sin(alpha)*cos(beta)*cos(theta_m/2 + pi/6) - r_m*sin(beta)*sin(theta_m/2 + pi/6))**2 + (P_x - r_f*sin(theta_f/2 + pi/6) + r_m*(sin(alpha)*sin(beta)*cos(gamma) - sin(gamma)*cos(alpha))*cos(theta_m/2 + pi/6) + r_m*sin(theta_m/2 + pi/6)*cos(beta)*cos(gamma))**2 + (P_y - r_f*cos(theta_f/2 + pi/6) + r_m*(sin(alpha)*sin(beta)*sin(gamma) + cos(alpha)*cos(gamma))*cos(theta_m/2 + pi/6) + r_m*sin(gamma)*sin(theta_m/2 + pi/6)*cos(beta))**2)],
[sqrt((P_z + r_m*sin(alpha)*sin(theta_m/2 + pi/3)*cos(beta) - r_m*sin(beta)*cos(theta_m/2 + pi/3))**2 + (P_x - r_f*cos(theta_f/2 + pi/3) + r_m*(sin(alpha)*sin(beta)*cos(gamma) - sin(gamma)*cos(alpha))*sin(theta_m/2 + pi/3) + r_m*cos(beta)*cos(gamma)*cos(theta_m/2 + pi/3))**2 + (P_y - r_f*sin(theta_f/2 + pi/3) + r_m*(sin(alpha)*sin(beta)*sin(gamma) + cos(alpha)*cos(gamma))*sin(theta_m/2 + pi/3) + r_m*sin(gamma)*cos(beta)*cos(theta_m/2 + pi/3))**2)],
[                                                        sqrt((P_z + r_m*sin(alpha)*s

In [7]:
values = {
    r_m: 10.0,
    r_f: 25.0,
    t_m: np.radians(30.0),
    t_f: np.radians(30.0),
    X: 0.0,
    Y: 0.0,
    Z: 20.0,
    alpha: np.radians(0.0),
    beta: np.radians(0.0),
    gamma: np.radians(0.0),
}

l1_num, l2_num, l3_num, l4_num, l5_num, l6_num = [i.subs(values) for i in (l_1, l_2, l_3, l_4, l_5, l_6)]

l = sp.Matrix([l1_num, l2_num, l3_num, l4_num, l5_num, l6_num])
l.evalf(6)

Matrix([
[25.0],
[25.0],
[25.0],
[25.0],
[25.0],
[25.0]])

### Forward Kinematics

### Loop closure equations

In [8]:
l1, l2, l3, l4, l5, l6 = sp.symbols('l_1, l_2, l_3, l_4, l_5, l_6', positive=True, real=True)

eq1 = (L1.dot(L1) - l1**2)
eq2 = (L2.dot(L2) - l2**2)
eq3 = (L3.dot(L3) - l3**2)
eq4 = (L4.dot(L4) - l4**2)
eq5 = (L5.dot(L5) - l5**2)
eq6 = (L6.dot(L6) - l6**2)


In [9]:
eq1.free_symbols

{P_x, P_y, P_z, alpha, beta, gamma, l_1, r_f, r_m, theta_f, theta_m}

In [10]:
values = {
    r_m: 10.0,
    r_f: 25.0,
    t_m: np.radians(30.0),
    t_f: np.radians(30.0),
    l1: 25.0,
    l2: 25.0,
    l3: 25.0,
    l4: 25.0,
    l5: 25.0,
    l6: 25.0
}

eq1_num, eq2_num, eq3_num, eq4_num, eq5_num, eq6_num = [i.subs(values) for i in (eq1, eq2, eq3, eq4, eq5, eq6)]
type(eq6_num)

sympy.core.add.Add

In [11]:
eq6_num.free_symbols

{P_x, P_y, P_z, alpha, beta, gamma}

### Newton Raphson numerical solution

In [14]:
platform_pose = sp.nsolve(
    [eq1_num, eq2_num, eq3_num, eq4_num, eq5_num, eq6_num],
    [X, Y, Z, alpha, beta, gamma],
    [0.00, 0.00, 19.9999, np.radians(0.00), np.radians(0.00), np.radians(0.00)],
    tol=1e-15,
    # verbose=True
)

labels = ["X", "Y", "Z", "alpha", "beta", "gamma"]

for label, i in zip(labels, platform_pose):
    value = float(i)
    if label in ("alpha", "beta", "gamma"):
        value = np.degrees(value)
    print(f"{label} = {value:.4f}")

X = -0.0001
Y = -0.0000
Z = 20.0000
alpha = -0.0000
beta = 0.0002
gamma = -0.0000
